# Entregable 2 — Modelos unimodales sobre sub50 y prototipo de fusión por votación

Proyecto Integrador (USFQ) · David Sebastián Bucheli Cabezas · Tutor: Daniel Riofrío

**Alcance (según la planificación):** modelos unimodales reentrenados sobre `sub50` y prototipo de la fusión por esquema de votación (actividades A5 y parte de A6).

**Este notebook se entrega sin salidas hasta que se ejecute con datos reales.** Ninguna cifra de este archivo es un resultado; todas se generan al correr las celdas. Ejecutar con *Restart & Run All* para que queden guardados los números de ejecución.

Contenido: 0) preparación · 1) exploración de datos · 2) entrenamiento en sub50 con checkpoints · 3) evaluación y matrices de confusión · 4) fusión por votación 2-de-3 · 5) extracción de embeddings (insumo del Entregable 3) · 6) resumen guardado en `results.json`.

## 0. Preparación

In [ ]:
import os, sys, json, time, platform
from pathlib import Path

# --- Dónde está el código --------------------------------------------------
# Este notebook asume que se ejecuta desde la raíz del repositorio (donde está la carpeta src/).
# Si lo corres en otra máquina sin el repo, pon aquí la URL de tu repositorio de GitHub.
REPO_URL = ""   # p. ej. "https://github.com/<usuario>/<repo>.git"
if not os.path.isdir("src"):
    if os.path.isdir("../src"):
        os.chdir("..")
    elif REPO_URL:
        os.system(f"git clone {REPO_URL} repo_clone")
        os.chdir("repo_clone")
    else:
        raise RuntimeError("No encuentro la carpeta src/. Ejecuta desde la raíz del repo o define REPO_URL.")
print("Directorio de trabajo:", os.getcwd())

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from src import config, datasets, models, engine, embeddings, audit, runs, fusion, explore, reporting

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
print("torch", torch.__version__, "| python", platform.python_version())

SUBSET = config.SUBSET_FORMAL          # "sub50"
OUT = Path("outputs_e2")               # resultados de esta corrida (checkpoints y .npz están en .gitignore)
for d in ("checkpoints", "predictions", "figures", "embeddings"):
    (OUT / d).mkdir(parents=True, exist_ok=True)

In [ ]:
# Login de Hugging Face: opcional (el dataset es público). Si lo usas, hazlo de forma interactiva; no pegues tokens en el notebook.
# from huggingface_hub import login; login()
ds = datasets.load_subset(SUBSET)
print(ds)

### Decisiones de protocolo (fijadas antes de ver resultados)

- **Mismo protocolo que el Entregable 1:** Adam, lr 1e-3, batch 32, 20 épocas, seed 42, los mismos modelos. El modelo que se reporta como principal es el de **la última época** (igual que en el Entregable 1), para que las cifras sean comparables.
- Además se guarda el checkpoint con **mejor macro-F1 en validación** (nunca se elige con test). Queda como variante complementaria; si se usa, se declara.
- **Votación:** mayoría 2-de-3. Con 10 clases los tres modelos pueden votar clases distintas; la regla de desempate se **elige con validación** (no con test) entre: `priority` (gana la modalidad con mejor macro-F1 de validación), `confidence` (gana el modelo más seguro) y `error` (sin decisión, cuenta como error).
- Métrica principal: **macro-F1** (más accuracy y reporte por clase), porque `sub50` probablemente está desbalanceado.
- Una sola corrida por modelo (seed 42): hay no determinismo entre corridas (el tutor obtuvo valores ligeramente distintos al reproducir el Entregable 1), así que diferencias pequeñas no son concluyentes.

## 1. Exploración de datos (sub50)

In [ ]:
splits = {k: ds[k] for k in ("train", "validation", "test")}
dist = explore.class_distribution(splits)
display(dist)
dist[["train", "validation", "test"]].plot(kind="bar", figsize=(10, 4), title=f"Objetos por clase — {SUBSET}")
plt.tight_layout(); plt.savefig(OUT / "figures" / "distribucion_clases.png", dpi=150); plt.show()
print("Cociente clase mayor / clase menor en train:", round(dist["train"].max() / dist["train"].min(), 2))

In [ ]:
lens = explore.photometry_lengths(ds["train"])
print(lens.describe())
print("Fracción de curvas con la longitud máxima:", round((lens == lens.max()).mean(), 3))
lens.plot(kind="hist", bins=40, figsize=(7, 3), title="Longitud de las curvas de luz (train)")
plt.tight_layout(); plt.savefig(OUT / "figures" / "longitud_curvas.png", dpi=150); plt.show()

In [ ]:
# ¿Qué son los 9 canales de la fotometría? Se mide, no se asume.
# Un canal 'constante en la curva' para (casi) todos los objetos es un valor por objeto (p. ej. período, amplitud),
# no una señal que cambia en el tiempo. Esto confirma o descarta la sospecha del tutor sobre los canales auxiliares.
phot_stats = explore.photometry_channel_stats(ds["train"], n=2000)
display(phot_stats)

In [ ]:
spec_stats = explore.spectra_channel_stats(ds["train"], n=1000)
display(spec_stats)   # escalas por canal, ceros, NaN: insumo para diagnosticar la CNN de espectros

In [ ]:
meta_stats = explore.metadata_stats(ds["train"])
display(meta_stats)
print("Columnas constantes:", int(meta_stats["constante"].sum()), "| NaN totales:", int(meta_stats["n_nan"].sum()))

In [ ]:
# Un ejemplo por clase (primer objeto de cada clase en train): primeros canales de la curva y espectro.
first = explore.first_example_per_class(ds["train"])
fig, axes = plt.subplots(len(first), 2, figsize=(11, 2.0 * len(first)))
for r, (c, idx) in enumerate(first.items()):
    row = ds["train"][idx]
    ph = np.asarray(row["photometry"]); sp = np.asarray(row["spectra"])
    for ch in range(min(3, ph.shape[1])):
        axes[r, 0].plot(ph[:, ch], lw=0.8, label=f"canal {ch}")
    axes[r, 0].set_ylabel(config.CLASS_NAMES[c])
    for ch in range(sp.shape[0]):
        axes[r, 1].plot(sp[ch], lw=0.6, label=f"canal {ch}")
axes[0, 0].set_title("Fotometría (canales 0-2 vs. índice)"); axes[0, 1].set_title("Espectro (3 canales)")
axes[0, 0].legend(fontsize=6); axes[0, 1].legend(fontsize=6)
plt.tight_layout(); plt.savefig(OUT / "figures" / "ejemplos_por_clase.png", dpi=130); plt.show()

**Anotar aquí lo que muestran las tablas** (después de ejecutar): qué canales de la fotometría son constantes por objeto, escalas de los 3 canales del espectro, columnas de metadatos constantes o con NaN, grado de desbalance. Esto cierra la deuda de exploración del Entregable 1 y orienta la revisión de la CNN (el tutor pidió revisarla): si los canales del espectro tienen escalas muy distintas o los espectros no están normalizados, es una hipótesis concreta; **no se modifica la CNN sin evidencia y sin registrarlo**.

## 2. Entrenamiento de los tres baselines sobre sub50 (con checkpoints)

In [ ]:
NAMES = ["fotometria", "espectros", "metadatos"]
FWD = {"fotometria": engine.forward_fn_photometry,
       "espectros": engine.forward_fn_spectra,
       "metadatos": engine.forward_fn_metadata}
META_DIM = len(ds["train"][0]["metadata"])

def build(name):
    torch.manual_seed(config.SEED)                      # misma semilla antes de construir cada modelo
    if name == "fotometria":
        enc = models.PhotometryGRUEncoder(input_dim=len(ds["train"][0]["photometry"][0]))
    elif name == "espectros":
        enc = models.SpectraCNNEncoder(in_channels=len(ds["train"][0]["spectra"]))
    else:
        enc = models.MetadataMLPEncoder(input_dim=META_DIM)
    return models.UnimodalClassifier(enc)

train_loader = datasets.make_dataloader(ds["train"], shuffle=True)
val_loader = datasets.make_dataloader(ds["validation"], shuffle=False)
fit_results, MODELS = {}, {}
for name in NAMES:
    print(f"\n===== {name} =====")
    torch.manual_seed(config.SEED)
    model = build(name)
    fit_results[name] = runs.fit_with_checkpoints(model, train_loader, val_loader, FWD[name],
                                                  device=DEVICE, class_names=config.CLASS_NAMES)
    runs.save_checkpoint(OUT / "checkpoints" / f"{name}_{SUBSET}.pt", fit_results[name],
                         extra={"seed": config.SEED, "subset": SUBSET})
    runs.save_history_json(OUT / f"history_{name}_{SUBSET}.json", fit_results[name]["history"],
                           extra={"train_seconds": fit_results[name]["train_seconds"],
                                  "n_parameters": fit_results[name]["n_parameters"],
                                  "best_epoch_val_macro_f1": fit_results[name]["best_epoch"]})
    MODELS[name] = model
    print(f"{name}: {fit_results[name]['train_seconds']:.0f} s | {fit_results[name]['n_parameters']:,} parámetros | mejor época (val macro-F1): {fit_results[name]['best_epoch']}")

In [ ]:
reporting.plot_history({n: fit_results[n]["history"] for n in NAMES}, path=OUT / "figures" / "curvas_entrenamiento.png")
plt.show()
cost = pd.DataFrame({n: {"parametros": fit_results[n]["n_parameters"],
                         "segundos_entrenamiento": round(fit_results[n]["train_seconds"], 1),
                         "mejor_epoca_val": fit_results[n]["best_epoch"]} for n in NAMES}).T
display(cost)    # costo de cómputo registrado (la planificación estimó ~30 h de GPU)

## 3. Evaluación en validación y test (modelo de la última época)

Se predice con `shuffle=False` para que las tres modalidades queden alineadas por índice. Se guardan probabilidades y predicciones.

In [ ]:
MODEL_SELECTION = "last"    # "last" = como el Entregable 1 (principal). "best" = mejor macro-F1 de validación (complementaria).
for n in NAMES:
    runs.load_checkpoint_into(MODELS[n], OUT / "checkpoints" / f"{n}_{SUBSET}.pt", which=MODEL_SELECTION, device=DEVICE)

loaders = {s: datasets.make_dataloader(ds[s], shuffle=False) for s in ("train", "validation", "test")}
PRED, METRICS = {"validation": {}, "test": {}}, {"validation": {}, "test": {}}
for split in ("validation", "test"):
    for n in NAMES:
        out = runs.predict_proba(MODELS[n], loaders[split], DEVICE, FWD[n])
        PRED[split][n] = out
        runs.save_predictions(OUT / "predictions" / f"{n}_{SUBSET}_{split}_{MODEL_SELECTION}.npz", out)
        METRICS[split][n] = fusion.summarize_predictions(out["y_true"], out["y_pred"], config.CLASS_NAMES)
    fusion.check_alignment([PRED[split][n]["y_true"] for n in NAMES])
print("Etiquetas alineadas entre las tres modalidades en validación y test.")
display(reporting.summary_table({f"{n} ({s})": METRICS[s][n] for s in ("validation", "test") for n in NAMES}))

In [ ]:
# Referencia: resultados del Entregable 1 (notebook ejecutado, test de sub50, entrenado en sub50, última época, 1 corrida).
E1_REF = {"fotometria": (0.886, 0.897), "metadatos": (0.820, 0.788), "espectros": (0.609, 0.478)}
cmp = pd.DataFrame({n: {"E1 accuracy": E1_REF[n][0], "E1 macro-F1": E1_REF[n][1],
                        "E2 accuracy": METRICS["test"][n]["accuracy"], "E2 macro-F1": METRICS["test"][n]["macro_f1"]}
                    for n in NAMES}).T.round(3)
display(cmp)   # diferencias pequeñas = no determinismo entre corridas, no mejora real

In [ ]:
for n in NAMES:
    print(f"\n===== {n} — test {SUBSET} =====")
    print(METRICS["test"][n]["report_text"])
    reporting.plot_confusion_matrix(METRICS["test"][n]["confusion_matrix"], title=f"{n} — test {SUBSET} (recall por fila)",
                                    path=OUT / "figures" / f"cm_{n}_{SUBSET}_test.png")
    plt.show()

## 4. Fusión por votación (2-de-3)

Fusión tardía: cada modelo unimodal emite su clase y se decide por mayoría. Si los tres difieren, se aplica la regla de desempate, **elegida con validación**.

In [ ]:
val_f1 = {n: METRICS["validation"][n]["macro_f1"] for n in NAMES}
PRIORITY = fusion.rank_by_validation(val_f1)
print("Prioridad por macro-F1 de validación:", PRIORITY)

def run_vote(split, rule):
    preds = {n: PRED[split][n]["y_pred"] for n in NAMES}
    probs = {n: PRED[split][n]["probs"] for n in NAMES}
    vote = fusion.hard_vote(preds, tie_break=rule, priority=PRIORITY, probs=probs)
    y_true = PRED[split][NAMES[0]]["y_true"]
    return vote, fusion.summarize_predictions(y_true, vote["y_pred"], config.CLASS_NAMES), y_true, preds

# 1) Elegir la regla de desempate SOLO con validación
val_scores = {}
for rule in fusion.TIE_BREAKS:
    _, m, _, _ = run_vote("validation", rule)
    val_scores[rule] = m["macro_f1"]
CHOSEN_RULE = max(val_scores, key=val_scores.get)
print("Macro-F1 de validación por regla de desempate:", {k: round(v, 4) for k, v in val_scores.items()})
print("Regla elegida (validación):", CHOSEN_RULE)

In [ ]:
# 2) Resultados en test: la regla elegida es la principal; las otras se muestran por transparencia.
VOTE, VOTE_METRICS = {}, {}
for rule in fusion.TIE_BREAKS:
    VOTE[rule], VOTE_METRICS[rule], y_true_test, preds_test = run_vote("test", rule)
tbl = reporting.summary_table({**{f"{n} (unimodal)": METRICS["test"][n] for n in NAMES},
                               **{f"votación 2-de-3, desempate={r}" + (" [elegida]" if r == CHOSEN_RULE else ""): VOTE_METRICS[r]
                                  for r in fusion.TIE_BREAKS}})
display(tbl)

In [ ]:
# Cuántas veces coinciden los modelos y cuánto acierta la votación en cada caso.
ag = fusion.agreement_analysis(y_true_test, preds_test, VOTE[CHOSEN_RULE])
print(json.dumps(ag, indent=2, ensure_ascii=False))

In [ ]:
print(VOTE_METRICS[CHOSEN_RULE]["report_text"])
reporting.plot_confusion_matrix(VOTE_METRICS[CHOSEN_RULE]["confusion_matrix"],
                                title=f"Votación 2-de-3 ({CHOSEN_RULE}) — test {SUBSET}", path=OUT / "figures" / f"cm_votacion_{SUBSET}_test.png")
plt.show()
f1_tbl = reporting.per_class_table({**{n: METRICS["test"][n] for n in NAMES}, "votación": VOTE_METRICS[CHOSEN_RULE]}, "f1").round(3)
display(f1_tbl)   # análisis por clase (objetivo específico v)

In [ ]:
# Referencia complementaria (NO es la regla 2-de-3 prometida): promedio de probabilidades.
soft_pred = fusion.soft_vote({n: PRED["test"][n]["probs"] for n in NAMES})
soft_m = fusion.summarize_predictions(y_true_test, soft_pred, config.CLASS_NAMES)
print(f"Voto suave (promedio de probabilidades) — test: accuracy={soft_m['accuracy']:.4f} macro-F1={soft_m['macro_f1']:.4f}")

## 5. Extracción de embeddings (insumo del Entregable 3)

Misma secuencia que en el Entregable 1 pero ahora ejecutada con los modelos entrenados. `shuffle=False` para alinear por índice. Nota metodológica: los embeddings de *train* salen de codificadores entrenados con esos mismos objetos, por lo que son «vistos»; hay que declararlo cuando se entrene la cabeza de concatenación.

In [ ]:
EMB = {s: {} for s in ("train", "validation", "test")}
for s in EMB:
    for n in NAMES:
        EMB[s][n] = embeddings.extract_embeddings(MODELS[n], loaders[s], DEVICE, FWD[n])
        embeddings.save_embeddings(str(OUT / "embeddings" / f"{n}_{SUBSET}_{s}.npz"), EMB[s][n]["embeddings"], EMB[s][n]["labels"])
    fusion.check_alignment([EMB[s][n]["labels"] for n in NAMES])
    fused = embeddings.concatenate_modalities(EMB[s]["fotometria"]["embeddings"], EMB[s]["espectros"]["embeddings"],
                                              EMB[s]["metadatos"]["embeddings"], EMB[s]["fotometria"]["labels"])
    embeddings.save_embeddings(str(OUT / "embeddings" / f"fused_{SUBSET}_{s}.npz"), fused["embeddings"], fused["labels"])
    print(s, "-> fusionado:", fused["embeddings"].shape, "(orden: fotometría | espectros | metadatos)")

## 6. Resumen guardado

In [ ]:
def clean(m):
    return {"accuracy": float(m["accuracy"]), "macro_f1": float(m["macro_f1"]),
            "per_class_f1": {c: float(v["f1"]) for c, v in m["per_class"].items()}}

results = {
    "subset": SUBSET, "seed": config.SEED, "model_selection": MODEL_SELECTION, "device": DEVICE,
    "torch": torch.__version__, "epochs": config.NUM_EPOCHS, "lr": config.LEARNING_RATE, "batch_size": config.BATCH_SIZE,
    "class_distribution_train": dist["train"].to_dict(),
    "cost": cost.to_dict(orient="index"),
    "unimodal": {s: {n: clean(METRICS[s][n]) for n in NAMES} for s in ("validation", "test")},
    "vote": {"priority_by_val": PRIORITY, "chosen_tie_break": CHOSEN_RULE,
             "val_macro_f1_by_rule": {k: float(v) for k, v in val_scores.items()},
             "test": {r: clean(VOTE_METRICS[r]) for r in fusion.TIE_BREAKS},
             "agreement_test": ag},
    "soft_vote_test": {"accuracy": float(soft_m["accuracy"]), "macro_f1": float(soft_m["macro_f1"])},
}
with open(OUT / "results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
print("Guardado:", OUT / "results.json")

**Fuera de este entregable (pendiente a propósito):** fusión por concatenación y transferencia desde AstroM3-CLIP (Entregable 3); ablación de canales de la fotometría y cuenta de FLOPs (se deja para el final).